# Smart Retail Analytics System

## Capstone Project – Computer Vision for Developers with Ultralytics

This project builds an end-to-end computer vision system for retail analytics using Ultralytics YOLO.

The system performs object detection and video analytics to support real-world retail scenarios such as customer counting, heatmap generation, privacy protection, and video analytics.

### Main Objectives

- Perform YOLO object detection.
- Apply a vision task beyond basic detection.
- Analyze customer movement using video analytics.
- Evaluate model performance using validation metrics.
- Fine-tune a YOLO model on a custom dataset.
- Export the trained model to an optimized format.

### Technologies

- Python
- Ultralytics YOLO
- OpenCV
- NumPy
- Google Colab


####**Developed by:** Saja Aljamal

## 1. Project Setup
### 1.1 Install Ultralytics

This section prepares the environment and installs the Ultralytics package required to build and run the computer vision pipeline.

In [ ]:
# Install Ultralytics
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.6 MB/s eta 0:00:00


### 1.2 Import Required Libraries

The following libraries are imported for model inference, video processing, data handling, and the implementation of Ultralytics solutions.

In [ ]:
from ultralytics import YOLO, solutions
import cv2
import os
import pandas as pd

print("Ultralytics and required libraries imported successfully!")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics and required libraries imported successfully!


## 2. YOLO Object Detection
### 2.1 Load YOLO Model

In this section, a pretrained YOLO model is loaded and used to perform object detection on an input image.
The model identifies objects in the image and returns bounding boxes, class labels, and confidence scores.
This step demonstrates the basic YOLO inference pipeline before applying the model to the retail analytics tasks.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO model
model = YOLO("yolo26n.pt")

print("Model loaded successfully!")

Model loaded successfully!


### Observation

The pretrained YOLO model was loaded successfully and is ready for inference.

### 2.2 Run YOLO Inference
The loaded YOLO model is now used to perform inference on an input image.
The resulting image will contain bounding boxes around detected objects, along with their class labels and confidence scores.

In [ ]:
import cv2

video_path = "solutions_ci_demo.mp4"
image_path = "detection_test_frame.jpg"

cap = cv2.VideoCapture(video_path)

success, frame = cap.read()

if not success:
    raise RuntimeError("Could not read the input video.")

cv2.imwrite(image_path, frame)

cap.release()

print("Test frame saved as:", image_path)

Test frame saved as: detection_test_frame.jpg


### Run Detection on the Extracted Frame
The extracted test frame is passed to the pretrained YOLO model to detect objects and generate bounding boxes, class labels, and confidence scores.

In [ ]:
# Run YOLO inference
results = model(image_path)

# Generate annotated image
annotated_image = results[0].plot()

# Save the detection result
detection_output = "detection_result.jpg"
cv2.imwrite(detection_output, annotated_image)

print("Detection completed successfully!")
print("Output saved as:", detection_output)


image 1/1 /content/detection_test_frame.jpg: 384x640 19 persons, 51.3ms
Speed: 73.6ms preprocess, 51.3ms inference, 6.1ms postprocess per image at shape (1, 3, 384, 640)
Detection completed successfully!
Output saved as: detection_result.jpg


### Observation

The YOLO model successfully detected objects in the test frame and generated an annotated image containing bounding boxes, class labels, and confidence scores.

This confirms that the basic YOLO object detection inference pipeline is working correctly.

## 3. Image Segmentation

In this section, YOLO segmentation is used as a vision task beyond basic object detection.

Unlike object detection, which represents an object using a bounding box, segmentation identifies the pixels belonging to each detected object using a segmentation mask.

This provides more detailed information about the shape and location of objects in the scene.

### 3.1 Load Segmentation Model

A pretrained YOLO segmentation model is loaded to perform instance segmentation on the test frame.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLO segmentation model
segmentation_model = YOLO("yolo26n-seg.pt")

print("Segmentation model loaded successfully!")

Segmentation model loaded successfully!


### 3.2 Run Segmentation

The same test frame used for object detection is passed to the YOLO segmentation model.

The model generates segmentation masks for the detected objects, providing a more detailed representation of each object than bounding boxes alone.

In [ ]:
# Run segmentation on the test frame
segmentation_results = segmentation_model(image_path)

# Generate the annotated segmentation image
segmented_image = segmentation_results[0].plot()

# Save the segmentation result
segmentation_output = "segmentation_result.jpg"
cv2.imwrite(segmentation_output, segmented_image)

print("Segmentation completed successfully!")
print("Output saved as:", segmentation_output)


image 1/1 /content/detection_test_frame.jpg: 384x640 12 persons, 42.7ms
Speed: 22.8ms preprocess, 42.7ms inference, 26.2ms postprocess per image at shape (1, 3, 384, 640)
Segmentation completed successfully!
Output saved as: segmentation_result.jpg


### Observation

The YOLO segmentation model successfully identified objects in the test frame and generated segmentation masks.

The masks provide more detailed spatial information about the detected objects compared with bounding boxes alone. This demonstrates a computer vision task beyond basic object detection.

## 4. Video Analytics

This section applies Ultralytics Solutions to the retail video to perform real-world video analytics.

The system includes customer counting, movement heatmap generation, privacy protection through object blurring, and video analytics.

### 4.1 People Counting

The YOLO ObjectCounter solution is used to track people and count customers crossing a defined horizontal line.

Only the person class is considered, and the BoT-SORT tracker is used to maintain object identities across video frames.

In [ ]:
from ultralytics import solutions
import cv2
import os

video_path = "solutions_ci_demo.mp4"
output_path = "people_counting_result.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

counter = solutions.ObjectCounter(
    model="yolo26n.pt",
    classes=[0],
    region=[(50, 200), (590, 200)],
    show=False,
    tracker="botsort.yaml"
)

frame_count = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    result = counter(frame)

    out.write(result.plot_im)

    frame_count += 1

cap.release()
out.release()

print("People counting completed!")
print("Frames processed:", frame_count)
print("Output saved as:", output_path)
print("IN:", counter.in_count)
print("OUT:", counter.out_count)

Ultralytics Solutions: ✅ {'source': None, 'model': 'yolo26n.pt', 'classes': [0], 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': [(50, 200), (590, 200)], 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 433ms
Prepared 1 package in 101ms
Installed 1 package in 3ms
 + lap==0

### Observation

The ObjectCounter successfully tracked people in the video and counted objects crossing the defined counting line.

The resulting video contains the detected people, bounding boxes, tracking information, and the IN/OUT counter.

### 4.2 Heatmap

The YOLO Heatmap solution is used to visualize areas with higher levels of person activity.

The heatmap helps identify frequently visited areas and can provide useful information about customer movement and popular zones inside a retail environment.

In [ ]:
from ultralytics import solutions
import cv2

video_path = "solutions_ci_demo.mp4"
output_path = "heatmap_result.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

heatmap = solutions.Heatmap(
    model="yolo26n.pt",
    classes=[0],
    show=False
)

frame_count = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    result = heatmap(frame)

    out.write(result.plot_im)

    frame_count += 1

cap.release()
out.release()

print("Heatmap completed!")
print("Frames processed:", frame_count)
print("Output saved as:", output_path)

Ultralytics Solutions: ✅ {'source': None, 'model': 'yolo26n.pt', 'classes': [0], 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 640x640 5.7ms, 18 person
Speed: 67.5ms track, 5.7ms solution per image at shape (1, 3, 640, 640)

1: 640x640 3.7ms, 18 person
Speed: 25.1ms track, 3.7ms solution per image at shape (1, 3, 640, 640)

2: 640x640 3.3ms, 16 person
Speed: 26.6ms track, 3.3ms solution per image at shape (1, 3, 640, 640)

3: 640x640 3.9m

### Observation

The Heatmap solution successfully visualized the movement of detected people throughout the video.

The resulting heatmap highlights areas where people were more frequently present or active, which can help identify popular zones in a retail environment.

### 4.3 Object Blurring

The YOLO ObjectBlurrer solution is used to blur detected people in the video.

This provides a simple privacy-preserving approach by obscuring customer identities while maintaining the usefulness of the video for analytics.

In [ ]:
from ultralytics import solutions
import cv2

video_path = "solutions_ci_demo.mp4"
output_path = "blurred_result.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

blurrer = solutions.ObjectBlurrer(
    model="yolo26n.pt",
    classes=[0],
    blur_ratio=0.5,
    show=False
)

frame_count = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    result = blurrer(frame)

    out.write(result.plot_im)

    frame_count += 1

cap.release()
out.release()

print("Object blurring completed!")
print("Frames processed:", frame_count)
print("Output saved as:", output_path)

Ultralytics Solutions: ✅ {'source': None, 'model': 'yolo26n.pt', 'classes': [0], 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 640x640 4.6ms, 18 person
Speed: 72.6ms track, 4.6ms solution per image at shape (1, 3, 640, 640)

1: 640x640 2.6ms, 18 person
Speed: 25.7ms track, 2.6ms solution per image at shape (1, 3, 640, 640)

2: 640x640 2.3ms, 16 person
Speed: 26.0ms track, 2.3ms solution per image at shape (1, 3, 640, 640)

3: 640x640 2.6m

### Observation

The ObjectBlurrer successfully detected and blurred people in the video.

The resulting video preserves the overall scene and object locations while reducing identifiable visual information, supporting privacy protection in a retail environment.

### 4.4 Video Analytics

The Ultralytics Analytics solution is used to analyze object movement across the video frames.

The analytics output provides a visual representation of object activity over time, which can support understanding of customer movement patterns in a retail environment.

In [ ]:
from ultralytics import solutions
import cv2

video_path = "solutions_ci_demo.mp4"
output_path = "analytics_result.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if fps <= 0:
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

out = cv2.VideoWriter(
    output_path,
    fourcc,
    fps,
    (width, height)
)

analytics = solutions.Analytics(
    model="yolo26n.pt",
    classes=[0],
    analytics_type="line",
    tracker="botsort.yaml",
    show=False
)

frame_number = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        break

    result = analytics(frame, frame_number)

    processed_frame = result.plot_im

    processed_frame = cv2.resize(
        processed_frame,
        (width, height)
    )

    out.write(processed_frame)

    frame_number += 1

cap.release()
out.release()

print("Analytics completed!")
print("Frames processed:", frame_number)
print("Output saved as:", output_path)

Ultralytics Solutions: ✅ {'source': None, 'model': 'yolo26n.pt', 'classes': [0], 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 640x640 119.8ms, 18 person
Speed: 112.6ms track, 119.8ms solution per image at shape (1, 3, 640, 640)

1: 640x640 0.4ms, 18 person
Speed: 39.1ms track, 0.4ms solution per image at shape (1, 3, 640, 640)

2: 640x640 0.3ms, 16 person
Speed: 37.6ms track, 0.3ms solution per image at shape (1, 3, 640, 640)

3: 640x640

### Observation

The Analytics solution successfully processed the video and generated a visual representation of object activity across frames.

The resulting video can be used to analyze customer movement patterns and activity within the retail environment.

## 5. Model Evaluation

This section evaluates the performance of the pretrained YOLO model using the validation dataset and standard object detection metrics.

The evaluation provides quantitative measurements that help assess how accurately the model detects objects.

### 5.1 Prepare Validation Dataset

A validation dataset is required to quantitatively evaluate the pretrained YOLO model.

The project uses a standard YOLO-format validation dataset so that the model can be evaluated using precision, recall, mAP50, and mAP50-95 metrics.

In [ ]:
import os
import yaml
from ultralytics.utils import ASSETS

print("Ultralytics assets directory:", ASSETS)

Ultralytics assets directory: /usr/local/lib/python3.12/dist-packages/ultralytics/assets


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

validation_results = model.val(
    data="coco8.yaml",
    imgsz=640,
    batch=8
)

print("Validation completed successfully!")

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.5 GFLOPs

WARNING ⚠️ Dataset 'coco8.yaml' images not found, missing path '/content/datasets/coco8/images/val'
Unzipping /content/datasets/coco8.zip to /content/datasets/coco8...: 100% ━━━━━━━━━━━━ 25/25 2.6Kfiles/s 0.0s
Dataset download success ✅ (0.3s), saved to /content/datasets

val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1166.3±573.3 MB/s, size: 54.0 KB)
val: Scanning /content/datasets/coco8/labels/val... 4 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4/4 90.1it/s 0.0s
val: New cache created: /content/datasets/coco8/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1it/s 0.9s
                   all          4         17      0.849      0.654      0.906      0.665
                person          3         10          1      0.427      

### 5.2 Evaluation Metrics

The validation results are used to measure the object detection performance of the pretrained YOLO model.

The main evaluation metrics are:

- Precision: measures how many predicted detections are correct.
- Recall: measures how many relevant objects were successfully detected.
- mAP50: mean Average Precision at an IoU threshold of 0.50.
- mAP50-95: mean Average Precision averaged across IoU thresholds from 0.50 to 0.95.

In [ ]:
print("Precision:", validation_results.box.mp)
print("Recall:", validation_results.box.mr)
print("mAP50:", validation_results.box.map50)
print("mAP50-95:", validation_results.box.map)

Precision: 0.8491325376834978
Recall: 0.6544623063673578
mAP50: 0.9064941894674905
mAP50-95: 0.6647706555170674


###5.3 Interpretation

The pretrained YOLO model was evaluated on the COCO8 validation dataset.

The model achieved a Precision of 0.849, meaning that most of the predicted detections were correct. The Recall of 0.654 indicates that the model detected a substantial portion of the objects present in the validation images, while some objects were missed, resulting in false negatives.

The mAP50 of 0.906 indicates strong detection performance when an IoU threshold of 0.50 is used. The mAP50-95 of 0.665 is lower because it evaluates detection quality across multiple IoU thresholds from 0.50 to 0.95 and is therefore more strict.

For the retail use case, false negatives may occur when people are partially occluded, small, or difficult to distinguish from the background. False positives may occur when background regions or other objects are incorrectly classified as people.

The confidence threshold controls how certain the model must be before accepting a detection, while the IoU threshold determines how much the predicted bounding box must overlap with the ground-truth box to be considered a correct detection.

These validation results provide a quantitative baseline for the pretrained model before custom training on a task-specific retail dataset.

### 5.4 Observation

The validation process completed successfully using the COCO8 dataset.

The results provide a quantitative baseline for the pretrained YOLO model. The model achieved strong Precision and mAP50 performance, while the lower Recall indicates that some objects were missed.

Since COCO8 is a general-purpose validation dataset, these results should be considered a baseline rather than a direct measurement of performance on the project's retail environment.